# Table of Contents

- Data Preparation
- Dealing with Outliers
- Correlation Analysis
  - Correlation Matrices
- Conceptual Models
  - Time-Series Forecast Models
  - Model Evaluation
- Regression Analysis
  - OLS Regression
- Exploring Data

# Data Preparation

## Load & Explore Raw Data

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

df = pd.read_csv('accounting_data.csv')
n1 = len(df)
df.head()

## Data Cleaning & Feature Engineering

In [ ]:
essential_cols = ['gvkey', 'fyear', 'at', 'ceq', 'ni', 'sale', 'csho']
df.dropna(subset=essential_cols, inplace=True)

df.sort_values(by=['gvkey', 'fyear'], inplace=True)
n2 = len(df)

df = df[(df['at'] > 0) & (df['ceq'] > 0) & (df['sale'] > 0) & (df['csho'] != 0)]
n3 = len(df)

# Dealing with Outliers

## Winsorization

In [ ]:
def wins(df, vars):
    def clip_series(s):
        lower = s.quantile(0.01)
        upper = s.quantile(0.99)
        return s.clip(lower=lower, upper=upper)
    for var in vars:
        df[var] = df.groupby('fyear')[var].transform(clip_series)
    return df

cols_to_winsorize = ['at', 'ni', 'sale', 'ceq', 'csho']
df = wins(df, cols_to_winsorize).dropna()  # if remove dropna() from this line, the result will be very different
n4 = len(df)

print("Yearly Winsorization complete.")

winsor_check = df.groupby('fyear')[['at']].agg(['min', 'max']).head(10)
display(winsor_check)

## Feature Engineering from Winsorized Data

In [ ]:
df['ni1'] = df.groupby('gvkey')['ni'].shift(-1)
df.dropna(subset=['ni1'], inplace=True)

df['roa_t1'] = df['ni1'] / df['at']
df['pm_t']  = df['ni'] / df['sale']
df['roa_t'] = df['ni'] / df['at']
df['log_at'] = np.log(df['at'])
df['loss_t'] = np.where(df['ni']<0, 1, 0)

df['eps'] = df['ni'] / df['csho']
df['eps_t1'] = df.groupby('gvkey')['eps'].shift(-1)
df['eps_t2'] = df.groupby('gvkey')['eps'].shift(-2)
df['eps_t3'] = df.groupby('gvkey')['eps'].shift(-3)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(subset=['roa_t1', 'pm_t', 'eps', 'eps_t1', 'eps_t2', 'eps_t3'], inplace=True)
n5 = len(df)

selection_summary = pd.DataFrame({
    'Filtering Step': [
        '1. Raw Data from accounting_data.csv',
        '2. Drop missing essential fields',
        '3. Economic Logic (at, ceq, sale > 0; csho != 0)',
        '4. Winsorize raw variables at 1%/99% by year',
        '5. Calculate ratios & EPS, remove Inf/NaN'
    ],
    'Observations Remaining': [n1, n2, n3, n4, n5],
    'Observations Dropped': [0, n1 - n2, n2 - n3, n3 - n4, n4 - n5]
})

selection_summary['Observations Remaining'] = selection_summary['Observations Remaining'].apply(lambda x: f'{x:,}')
selection_summary['Observations Dropped'] = selection_summary['Observations Dropped'].apply(lambda x: f'{x:,}')

display(selection_summary)

## Ratio Winsorization

In [ ]:
ENABLE_RATIO_WINSOR = True
if ENABLE_RATIO_WINSOR:
    print("Before:")
    from IPython.display import display
    display(df[['roa_t1','pm_t','roa_t']].describe())

    ratio_vars = ['roa_t1', 'pm_t', 'roa_t']
    for var in ratio_vars:
        df[var] = df.groupby('fyear')[var].transform(
            lambda s: s.clip(s.quantile(0.01), s.quantile(0.99))
        )

    print("After:")
    display(df[['roa_t1','pm_t','roa_t']].describe())

# Correlation Analysis

## Descriptive Statistics

In [ ]:
desc = df[['roa_t1', 'pm_t', 'eps', 'eps_t1', 'eps_t2', 'eps_t3']].describe()
display(desc)

## Time Trend of Profit Margin

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

_, ax = plt.subplots(figsize=(16, 8))
sns.lineplot(data=df, x='fyear', y='pm_t', marker='o', ax=ax)
plt.show()

## Correlation Matrices

### Pearson Correlation

In [ ]:
from scipy.stats import pearsonr

corr_vars = ['roa_t1', 'roa_t', 'pm_t', 'log_at', 'loss_t', 'eps']

rho = df[corr_vars].corr()

mask = np.triu(np.ones_like(rho, dtype=bool))
plt.figure(figsize=(10, 8))
sns.heatmap(rho, mask=mask, annot=True, cmap='Reds', fmt='.2f')
plt.title('Pearson Correlation Matrix')
plt.show()

pval = df[corr_vars].corr(
    method=lambda x, y: pearsonr(x, y)[1]
)

np.fill_diagonal(pval.values, 1)

def significance_stars(p):
    if p <= 0.001:
        return '***'
    elif p <= 0.01:
        return '**'
    elif p <= 0.05:
        return '*'
    return ''

stars = pval.map(significance_stars)

rho_stars = rho.round(2).astype(str) + stars
rho_stars = rho_stars.where(~mask, '')
display(rho_stars)

### Spearman Correlation

In [ ]:
from scipy.stats import spearmanr

corr2 = df[corr_vars].corr(method='spearman')

plt.figure(figsize=(10, 8))
sns.heatmap(corr2, mask=mask, cmap='Reds', annot=True, fmt='.2f')
plt.title('Spearman Correlation Matrix')
plt.show()

pval_spearman = df[corr_vars].corr(
    method=lambda x, y: spearmanr(x, y)[1]
)
np.fill_diagonal(pval_spearman.values, 1)

stars_spearman = pval_spearman.map(significance_stars)
corr2_stars = corr2.round(2).astype(str) + stars_spearman
corr2_stars = corr2_stars.where(~mask, '')
display(corr2_stars)

# Conceptual Models

## Time-Series Forecast Models

### Random Walk Baseline

In [ ]:
df['error_rw'] = abs(df['ni1'] - df['ni']) / df['at']
df.head()

### Multi-period RW & Moving Average

In [ ]:
df['error_rw_1'] = abs(df['eps_t1'] - df['eps'])
df['error_rw_2'] = abs(df['eps_t2'] - df['eps'])
df['error_rw_3'] = abs(df['eps_t3'] - df['eps'])

grouped_eps = df.groupby('gvkey')['eps']

def calc_moving_avg(eps_series, window_size):
    return eps_series.rolling(window=window_size, min_periods=window_size).mean()

df['ma_2'] = grouped_eps.transform(calc_moving_avg, window_size=2)
df['ma_3'] = grouped_eps.transform(calc_moving_avg, window_size=3)
df['ma_5'] = grouped_eps.transform(calc_moving_avg, window_size=5)

df['error_ma_2'] = abs(df['eps_t1'] - df['ma_2'])
df['error_ma_3'] = abs(df['eps_t1'] - df['ma_3'])
df['error_ma_5'] = abs(df['eps_t1'] - df['ma_5'])

error_cols = ['error_rw_1', 'error_rw_2', 'error_rw_3', 'error_ma_2', 'error_ma_3', 'error_ma_5']
df_errors = df.dropna(subset=error_cols)

error_medians = df_errors[error_cols].median()

plt.figure(figsize=(10, 6))
sns.barplot(x=error_medians.index, y=error_medians.values, palette='viridis', hue=error_medians.index, legend=False)
plt.title('Median Absolute Forecast Error by Model')
plt.xlabel('Model')
plt.ylabel('Median Absolute Forecast Error')
plt.show()

## Model Evaluation

### Sub-sample Analysis by Firm Size

In [ ]:
median_at_per_year = df.groupby('fyear')['at'].transform('median')
df['size_group'] = np.where(df['at'] >= median_at_per_year, 'Large', 'Small')

all_error_cols = ['error_rw_1', 'error_rw_2', 'error_rw_3',
                  'error_ma_2', 'error_ma_3', 'error_ma_5']
sub_df = df.dropna(subset=all_error_cols + ['size_group'])

median_errors = sub_df.groupby('size_group')[all_error_cols].median()

median_errors_reset = median_errors.reset_index().melt(
    id_vars='size_group', var_name='Model', value_name='Absolute Error'
)

plt.figure(figsize=(10, 6))
sns.barplot(data=median_errors_reset, x='Model', y='Absolute Error',
            hue='size_group', palette='Set2')
plt.title('Forecast Error Comparison: Large vs Small Firms (All Models)')
plt.xlabel('Model')
plt.ylabel('Median Absolute Forecast Error')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Scaled Forecast Error Analysis

In [ ]:
df['scaled_error_rw_1'] = abs(df['eps_t1'] - df['eps']) / abs(df['eps'])
df['scaled_error_rw_2'] = abs(df['eps_t2'] - df['eps']) / abs(df['eps'])
df['scaled_error_rw_3'] = abs(df['eps_t3'] - df['eps']) / abs(df['eps'])

df['scaled_error_ma_2'] = abs(df['eps_t1'] - df['ma_2']) / abs(df['eps'])
df['scaled_error_ma_3'] = abs(df['eps_t1'] - df['ma_3']) / abs(df['eps'])
df['scaled_error_ma_5'] = abs(df['eps_t1'] - df['ma_5']) / abs(df['eps'])

scaled_cols = ['scaled_error_rw_1', 'scaled_error_rw_2', 'scaled_error_rw_3', 
               'scaled_error_ma_2', 'scaled_error_ma_3', 'scaled_error_ma_5']
df[scaled_cols] = df[scaled_cols].replace([np.inf, -np.inf], np.nan)

sub_df_scaled = df.dropna(subset=scaled_cols + ['size_group'])

median_scaled_errors = sub_df_scaled.groupby('size_group')[scaled_cols].median()

median_scaled_errors_reset = median_scaled_errors.reset_index().melt(
    id_vars='size_group', var_name='Model', value_name='Scaled Absolute Error'
)

plt.figure(figsize=(10, 6))
sns.barplot(data=median_scaled_errors_reset, x='Model', y='Scaled Absolute Error',
            hue='size_group', palette='Set2')
plt.title('Forecast Error Comparison: Large vs Small Firms (Scaled/Standardized)')
plt.xlabel('Model')
plt.ylabel('Median Scaled Absolute Forecast Error')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Regression Analysis

## OLS Regression

### Interpret the coefficients

### Understanding the output from regressions

In [ ]:
import statsmodels.formula.api as smf

formula_hybrid = "roa_t1 ~ roa_t + pm_t + loss_t + log_at"
model_hybrid = smf.ols(formula_hybrid, data=df).fit()
print("  Hybrid Model: roa_t1 ~ roa_t + pm_t + loss_t + log_at")
model_hybrid.summary()

# Exploring Data

## Forecasting ROE and Earnings

In [ ]:
df["pred_hybrid"]  = model_hybrid.predict(df)

df["error_hybrid"]  = abs(df["roa_t1"] - df["pred_hybrid"])

error_full = pd.DataFrame({
    "Model":     ["Hybrid", "Random Walk"],
    "Median AE": [df["error_hybrid"].median(),
                  df["error_rw"].median()],
    "Mean AE":   [df["error_hybrid"].mean(),
                  df["error_rw"].mean()]
})
display(error_full)

## Comparing the forecast errors

In [ ]:
pre_2010  = df[df["fyear"] < 2010]
post_2010 = df[df["fyear"] >= 2010]

print(f"Pre-2010 observations:  {len(pre_2010):,}")
print(f"Post-2010 observations: {len(post_2010):,}")

period_errors = pd.DataFrame({
    "Model":     ["Hybrid", "Random Walk",
                  "Hybrid", "Random Walk"],
    "Period":    ["Pre-2010", "Pre-2010",
                  "Post-2010", "Post-2010"],
    "Median AE": [
        pre_2010["error_hybrid"].median(),
        pre_2010["error_rw"].median(),
        post_2010["error_hybrid"].median(),
        post_2010["error_rw"].median()
    ]
})

### Model Comparison Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=error_full, x="Model", y="Median AE",
            ax=axes[0], palette="Set2", hue="Model", legend=False)
axes[0].set_title("Median Absolute Error (Full Sample)")
axes[0].set_ylabel("Median AE")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=period_errors, x="Model", y="Median AE",
            hue="Period", ax=axes[1], palette="Set2")
axes[1].set_title("Median Absolute Error by Period")
axes[1].set_ylabel("Median AE")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()